# 01 — Paper discovery and metadata

Owner: Member 1. Finds every paper in the assigned ID range, works out how many versions
each one has, and stores per-version metadata in `<paper-folder>/metadata.json`.

Functions provided to the rest of the pipeline (agreed interface):

| Function | Returns |
|---|---|
| `list_ids(start, end)` | `list[str]` of arXiv IDs in the range (inclusive) |
| `fetch_metadata(ids)` | `list[dict]`, one per requested ID: `id, status, versions, title, authors, submitted, revised, venue, version_info` |
| `save_metadata(paper)` | writes `metadata.json`, returns its path (`None` if the paper was not found) |
| `load_metadata(arxiv_id)` | the saved dict, or `None` |

`status` is `ok` (all versions found), `partial` (some older version could not be fetched),
`not_found` (ID does not exist on arXiv) or `error` (the request kept failing).

### How it works

1. **Pass 1, latest versions.** Query the arXiv API with up to 100 plain IDs per call
   (`id_list`). arXiv returns the *latest* version of each paper, e.g. `1706.03762v7`, so the
   suffix gives the number of versions. IDs that do not exist are simply missing from the reply.
2. **Pass 2, older versions.** Query the versioned IDs `v1 … v(n-1)`, again 100 per call.
   For a versioned ID the API returns that version's own title, authors, abstract and date
   (`updated`), which gives the revision dates and per-version metadata.
3. Batching keeps the number of calls low: with the mandatory 3 s pause, 1,000 papers need
   about 10 calls in pass 1 instead of 1,000.

**Error handling.** The arXiv API sometimes answers a batch with HTTP 500 even though each ID
in it is valid. After `arxiv.py`'s own retries, a failing batch is split in half and each half
is retried, down to single IDs; an ID that still fails is reported with status `error`.

Library: `arxiv.py` 4.x, used only through `Client.results(Search(...))` as the lab book requires.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import arxiv
import requests

## 1. Build the ID list

In [ ]:
def list_ids(start, end):
    """All arXiv IDs from start to end (inclusive), e.g. list_ids('2310.00001', '2310.00003').

    Works for new-style IDs (same YYMM) and old-style IDs (same archive/YYMM),
    keeping the zero-padding of the input.
    """
    new_a, new_b = NEW_ID_RE.match(start), NEW_ID_RE.match(end)
    old_a, old_b = OLD_ID_RE.match(start), OLD_ID_RE.match(end)

    if new_a and new_b:
        if new_a.group(1) != new_b.group(1):
            raise ValueError("start and end must have the same YYMM prefix")
        prefix = new_a.group(1) + "."
        width = len(new_a.group(2))
        lo, hi = int(new_a.group(2)), int(new_b.group(2))
    elif old_a and old_b:
        if old_a.group(1) != old_b.group(1) or old_a.group(2)[:4] != old_b.group(2)[:4]:
            raise ValueError("start and end must have the same archive/YYMM prefix")
        prefix = f"{old_a.group(1)}/{old_a.group(2)[:4]}"
        width = 3
        lo, hi = int(old_a.group(2)[4:]), int(old_b.group(2)[4:])
    else:
        raise ValueError(f"unrecognised ID format: {start!r}, {end!r}")

    if lo > hi:
        raise ValueError("start must not be after end")
    return [f"{prefix}{n:0{width}d}" for n in range(lo, hi + 1)]

## 2. Query the arXiv API

One shared `Client` keeps the 3-second pause between *all* discovery requests.
`DISCOVERY_STATS` collects numbers for the report (calls made, IDs that failed, time spent).

In [ ]:
ARXIV_CLIENT = arxiv.Client(
    page_size=ARXIV_BATCH_SIZE,
    delay_seconds=ARXIV_DELAY,
    num_retries=ARXIV_RETRIES,
)

DISCOVERY_STATS = {"queries": 0, "failed_ids": [], "seconds": 0.0}


def _chunks(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def _query_arxiv(id_list):
    """arxiv.Result objects for id_list; IDs that do not exist are absent from the result.

    If the batch keeps failing, split it in half and retry each half,
    so one problematic ID cannot block the other 99.
    """
    DISCOVERY_STATS["queries"] += 1
    try:
        search = arxiv.Search(id_list=id_list, max_results=len(id_list))
        return list(ARXIV_CLIENT.results(search))
    except (arxiv.HTTPError, arxiv.UnexpectedEmptyPageError, requests.RequestException) as err:
        if len(id_list) == 1:
            print(f"[discovery] giving up on {id_list[0]}: {err}")
            DISCOVERY_STATS["failed_ids"].append(id_list[0])
            return []
        mid = len(id_list) // 2
        return _query_arxiv(id_list[:mid]) + _query_arxiv(id_list[mid:])

## 3. Turn API results into metadata records

In [ ]:
def _clean(text):
    """Collapse the line breaks and double spaces arXiv leaves in titles and abstracts."""
    return " ".join(text.split()) if text else text


def _version_record(result):
    """Metadata of one version, taken from the API entry for that version."""
    _, version = split_version(result.get_short_id())
    return {
        "version": version,
        "date": to_iso(result.updated),   # submission date of this version
        "title": _clean(result.title),
        "authors": [a.name for a in result.authors],
        "abstract": _clean(result.summary),
        "journal_ref": result.journal_ref,
        "doi": result.doi,
        "comment": result.comment,
        "categories": result.categories,
    }


def fetch_metadata(ids):
    """Metadata of every version of every paper in ids (one dict per distinct input ID)."""
    start = time.perf_counter()
    # Version suffixes in the input are ignored: we always want all versions
    ids = list(dict.fromkeys(split_version(i)[0] for i in ids))

    # Pass 1: latest version of each paper -> number of versions
    latest = {}
    for batch in _chunks(ids, ARXIV_BATCH_SIZE):
        for result in _query_arxiv(batch):
            base, n_versions = split_version(result.get_short_id())
            latest[base] = (n_versions, result)

    # Pass 2: every older version, by its versioned ID
    wanted = [f"{base}v{v}" for base, (n, _) in latest.items() for v in range(1, n)]
    older = {}
    for batch in _chunks(wanted, ARXIV_BATCH_SIZE):
        for result in _query_arxiv(batch):
            older[result.get_short_id()] = result

    failed = set(DISCOVERY_STATS["failed_ids"])
    papers = []
    for base in ids:
        if base not in latest:
            papers.append({"id": base, "status": "error" if base in failed else "not_found",
                           "versions": 0})
            continue

        n_versions, newest = latest[base]
        version_info = []
        for v in range(1, n_versions + 1):
            result = newest if v == n_versions else older.get(f"{base}v{v}")
            # A version whose query failed keeps an empty record instead of being dropped
            version_info.append(_version_record(result) if result else {"version": v, "date": None})

        papers.append({
            "id": base,
            "status": "ok" if all(r["date"] for r in version_info) else "partial",
            "versions": n_versions,
            "title": _clean(newest.title),
            "authors": [a.name for a in newest.authors],
            "submitted": to_iso(newest.published),               # date of v1
            "revised": [r["date"] for r in version_info[1:]],    # dates of v2..vN
            "venue": newest.journal_ref,                         # None if not published
            "version_info": version_info,
            "fetched_at": to_iso(datetime.now(timezone.utc)),
        })

    DISCOVERY_STATS["seconds"] += time.perf_counter() - start
    return papers

## 4. Save and load `metadata.json`

In [ ]:
def save_metadata(paper):
    """Write <paper-folder>/metadata.json. Papers that do not exist on arXiv get no folder."""
    if paper["status"] in ("not_found", "error"):
        return None
    path = paper_dir(paper["id"]) / "metadata.json"
    write_json(path, paper)
    return path


def load_metadata(arxiv_id):
    return read_json(DATA_DIR / to_folder_id(arxiv_id) / "metadata.json")

## 5. Quick test

Runs only when this notebook is opened on its own (`RUN_TEST = True`). The sample covers a
paper with many versions (`1706.03762`, 7 versions), an old-style ID (`hep-th/9711200`), a
paper with a journal reference, and an ID that does not exist (`2310.99999`).

In [ ]:
if RUN_TEST:
    # Offline checks of the ID helpers
    assert list_ids("2310.00098", "2310.00101") == ["2310.00098", "2310.00099", "2310.00100", "2310.00101"]
    assert list_ids("1401.0001", "1401.0002") == ["1401.0001", "1401.0002"]
    assert list_ids("hep-th/9901001", "hep-th/9901002") == ["hep-th/9901001", "hep-th/9901002"]
    assert split_version("2310.12345v2") == ("2310.12345", 2)
    assert to_folder_id("hep-th/9901001v2") == "hep-th_9901001"

    TEST_IDS = [
        "1706.03762", "hep-th/9711200", "1512.03385", "1810.04805", "2005.14165",
        "1409.0473", "2106.09685", "2201.11903", "2310.06825", "2310.99999",
    ]
    test_papers = fetch_metadata(TEST_IDS)
    for p in test_papers:
        save_metadata(p)
        print(f"{p['id']:<16} {p['status']:<10} versions={p['versions']}  "
              f"revised={len(p.get('revised', []))}  venue={p.get('venue')}")
    print(DISCOVERY_STATS)